In [3]:

import numpy as np
import pandas as pd

# Make the experiment reproducible
rng = np.random.default_rng(42)

# Generate 200 synthetic patients and 5,000 random features
n_patients = 200
n_features = 5000

X = rng.normal(size=(n_patients, n_features))

# Random labels: 0 = negative, 1 = positive
# These labels are independent of every feature
y = rng.integers(0, 2, size=n_patients)

print("Patients:", X.shape[0])
print("Random features:", X.shape[1])
print("Positive labels:", y.sum())
print("Negative labels:", (y == 0).sum())


Patients: 200
Random features: 5000
Positive labels: 95
Negative labels: 105


In [4]:

# Correlation between each feature and the random label
correlations = np.array([
    np.corrcoef(X[:, j], y)[0, 1]
    for j in range(X.shape[1])
])

print("Largest absolute feature-label correlation:",
      np.nanmax(np.abs(correlations)))

print("Number of features:", len(correlations))


Largest absolute feature-label correlation: 0.2815878648264217
Number of features: 5000


In [5]:

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# FLAWED METHOD:
# Select the 100 best features using ALL the labels first.
# This leaks information from future validation folds.
selector = SelectKBest(score_func=f_classif, k=100)
X_selected = selector.fit_transform(X, y)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

auc_scores = cross_val_score(
    model, X_selected, y,
    cv=cv, scoring="roc_auc"
)

print("Flawed AUC scores:", auc_scores)
print("Mean flawed AUC:", round(auc_scores.mean(), 4))


Flawed AUC scores: [0.98245614 0.99749373 0.9924812  0.98746867 0.97744361]
Mean flawed AUC: 0.9875


In [6]:

# CORRECT METHOD:
# Feature selection now happens inside each training fold.
correct_model = make_pipeline(
    SelectKBest(score_func=f_classif, k=100),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

correct_auc_scores = cross_val_score(
    correct_model, X, y,
    cv=cv, scoring="roc_auc"
)

print("Correct AUC scores:", correct_auc_scores)
print("Mean correct AUC:", round(correct_auc_scores.mean(), 4))


Correct AUC scores: [0.64160401 0.47117794 0.54887218 0.46115288 0.50626566]
Mean correct AUC: 0.5258


In [7]:

results = pd.DataFrame({
    "Method": [
        "Flawed: selection before cross-validation",
        "Correct: selection inside cross-validation"
    ],
    "Mean ROC-AUC": [
        auc_scores.mean(),
        correct_auc_scores.mean()
    ]
})

print(results.to_string(index=False))
print("\nRandom-guessing reference AUC: 0.5000")


                                    Method  Mean ROC-AUC
 Flawed: selection before cross-validation      0.987469
Correct: selection inside cross-validation      0.525815

Random-guessing reference AUC: 0.5000


In [8]:

from sklearn.base import clone
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

results = []

for seed in range(10):
    rng = np.random.default_rng(seed)
    X_repeat = rng.normal(size=(200, 5000))
    y_repeat = rng.integers(0, 2, size=200)

    cv_repeat = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=42
    )

    # Flawed: select features using all labels
    X_selected = SelectKBest(
        f_classif, k=100
    ).fit_transform(X_repeat, y_repeat)

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000)
    )

    flawed = cross_val_score(
        model, X_selected, y_repeat,
        cv=cv_repeat, scoring="roc_auc"
    ).mean()

    # Correct: selection occurs within each fold
    correct_model = make_pipeline(
        SelectKBest(f_classif, k=100),
        StandardScaler(),
        LogisticRegression(max_iter=2000)
    )

    correct = cross_val_score(
        correct_model, X_repeat, y_repeat,
        cv=cv_repeat, scoring="roc_auc"
    ).mean()

    results.append({
        "Seed": seed,
        "Flawed AUC": flawed,
        "Correct AUC": correct
    })

df_results = pd.DataFrame(results)
print(df_results.round(4).to_string(index=False))
print("\nMean AUC across seeds:")
print(df_results[["Flawed AUC", "Correct AUC"]].mean().round(4))


 Seed  Flawed AUC  Correct AUC
    0      0.9960       0.5752
    1      0.9409       0.5193
    2      0.9534       0.4727
    3      0.9865       0.4737
    4      0.9820       0.5338
    5      0.9654       0.4568
    6      0.9760       0.4634
    7      0.9920       0.5229
    8      0.9840       0.5135
    9      0.9834       0.6002

Mean AUC across seeds:
Flawed AUC     0.9759
Correct AUC    0.5132
dtype: float64
